# Guide: Cross-Currency Curves and Collateral Effects

Calibrates HUF/USD cross-currency basis curves and compares valuation impacts under a devaluation/funding stress regime.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

sys.path.append(str(Path.cwd().resolve().parent))

from src.curves.cross_currency import CurveInstrumentSet, calibrate_cross_currency_bundle, discount_factor


In [ ]:
tenors = [0.25, 0.5, 1.0, 2.0, 3.0]
ois_huf = {t: 0.07 - 0.002*t for t in tenors}
ois_usd = {t: 0.05 - 0.0015*t for t in tenors}
ois_eur = {t: 0.03 - 0.001*t for t in tenors}
spot = 360.0
basis_quotes = {t: 0.001 + 0.0003*t for t in tenors}

df = lambda r,t: 1/(1+r*t)
fwds = {t: spot*df(ois_huf[t],t)/(df(ois_usd[t],t)*np.exp(-basis_quotes[t]*t)) for t in tenors}

base = CurveInstrumentSet(
    ois_by_ccy={'HUF':ois_huf,'USD':ois_usd,'EUR':ois_eur},
    irs_by_ccy={'HUF':ois_huf,'USD':ois_usd,'EUR':ois_eur},
    fx_spot={'HUF/USD':spot},
    fx_forwards={'HUF/USD':fwds},
    xccy_basis_by_pair={'HUF/USD':basis_quotes},
)
base_bundle = calibrate_cross_currency_bundle(base)

# Stress: weaker HUF spot + wider basis quotes.
stress_spot = 392.0
stress_basis_quotes = {t: b + 0.0015 for t,b in basis_quotes.items()}
stress_fwds = {t: stress_spot*df(ois_huf[t]+0.004,t)/(df(ois_usd[t]+0.001,t)*np.exp(-stress_basis_quotes[t]*t)) for t in tenors}
stress = CurveInstrumentSet(
    ois_by_ccy={'HUF':{t:ois_huf[t]+0.004 for t in tenors},'USD':{t:ois_usd[t]+0.001 for t in tenors},'EUR':ois_eur},
    irs_by_ccy={'HUF':{t:ois_huf[t]+0.004 for t in tenors},'USD':{t:ois_usd[t]+0.001 for t in tenors},'EUR':ois_eur},
    fx_spot={'HUF/USD':stress_spot},
    fx_forwards={'HUF/USD':stress_fwds},
    xccy_basis_by_pair={'HUF/USD':stress_basis_quotes},
)
stress_bundle = calibrate_cross_currency_bundle(stress)

compare_basis = pd.DataFrame({
    'tenor': tenors,
    'base_basis': [base_bundle.basis_term_structures['HUF-USD'][t] for t in tenors],
    'stress_basis': [stress_bundle.basis_term_structures['HUF-USD'][t] for t in tenors],
})
compare_basis['delta_bp'] = (compare_basis['stress_basis']-compare_basis['base_basis'])*1e4
compare_basis


In [ ]:
cashflow = 100_000_000
t = 2.0
pv_base_huf_coll = cashflow * discount_factor(base_bundle, 'HUF', t, 'HUF')
pv_base_usd_coll = cashflow * discount_factor(base_bundle, 'HUF', t, 'USD')
pv_stress_usd_coll = cashflow * discount_factor(stress_bundle, 'HUF', t, 'USD')

pv_table = pd.DataFrame({
    'metric':['PV base (HUF coll)','PV base (USD coll)','PV stress (USD coll)','Stress minus base USD-coll PV'],
    'value':[pv_base_huf_coll,pv_base_usd_coll,pv_stress_usd_coll,pv_stress_usd_coll-pv_base_usd_coll]
})
pv_table


In [ ]:
class CrossCurrencyExplainer:
    def __init__(self, compare_basis: pd.DataFrame, pv_table: pd.DataFrame):
        self.compare_basis = compare_basis
        self.pv_table = pv_table

    def render_full_markdown(self) -> str:
        avg_basis_widen = self.compare_basis['delta_bp'].mean()
        pv_delta = float(self.pv_table.loc[self.pv_table['metric']=='Stress minus base USD-coll PV','value'].item())
        return f"""
### Interpretation (P&L / DV01 / Convexity / Basis / Hedge Action)
- **What changed:** cross-currency basis widens by about **{avg_basis_widen:.1f} bp** on average under stress.
- **P&L:** USD-collateralized PV changes by roughly **{pv_delta:,.0f}**, reflecting funding + rates regime shift.
- **DV01:** collateral currency switch changes effective rate sensitivity and hedge ratios.
- **Convexity:** nonlinearity appears when spot, basis, and discount curves move jointly.
- **Basis:** basis is the dominant incremental driver versus single-curve valuation.
- **Hedge action:** pair IRS DV01 hedges with xccy basis swaps/FX forwards to neutralize collateral and basis exposures.
""".strip()

explainer = CrossCurrencyExplainer(compare_basis, pv_table)
print(explainer.render_full_markdown())
